## Void Striker

Destroy enemies to unlock a walkthrough of how this game was built.
Every system in Void Striker — the scrolling starfield, the scene switcher, the shooting, the ship selection, the boss AI, and the gravity physics — is a technique you will learn by playing through it.

**Shoot 5 enemies** to unlock the next lesson block. Keep going until all 4 sections are revealed.

### Controls

| Key | Action |
|---|---|
| WASD | Move your ship |
| Arrow Keys | Fire (triple-shot spread) |
| Background button (top center) | Cycle between Nebula, Deep Space, and Supernova scenes |
| P | Pause and open ship selection menu |


In [ ]:
%%js

// GAME_RUNNER: Void Striker | hide_edit: true, width: 100%, height: 500px

import GameControl from '/assets/js/GameEnginev1.1/essentials/GameControl.js';
import GameLevelVoidStriker from '/assets/js/projects/voidstriker/levels/GameLevelVoidStriker.js';
export const gameLevelClasses = [GameLevelVoidStriker];
export { GameControl };


<div id='lesson-progress-bar' style='
  display: flex;
  flex-direction: row;
  gap: 8px;
  align-items: center;
  font-family: monospace;
  background: #0a001a;
  border: 1px solid #4400aa;
  padding: 10px 14px;
  border-radius: 6px;
  margin-bottom: 10px;
  flex-wrap: wrap;
'>
  <span style='color:#aaaaff; font-size:13px; margin-right:6px;'>Progress:</span>
  <div id='sq-1' style='width:90px; height:50px; border:2px solid #4400aa; border-radius:5px; display:flex; align-items:center; justify-content:center; font-size:11px; color:#553388; background:#0d0020; text-align:center; padding:4px;'>🔒<br>Parallax BG</div>
  <div id='sq-2' style='width:90px; height:50px; border:2px solid #4400aa; border-radius:5px; display:flex; align-items:center; justify-content:center; font-size:11px; color:#553388; background:#0d0020; text-align:center; padding:4px;'>🔒<br>BG Change</div>
  <div id='sq-3' style='width:90px; height:50px; border:2px solid #4400aa; border-radius:5px; display:flex; align-items:center; justify-content:center; font-size:11px; color:#553388; background:#0d0020; text-align:center; padding:4px;'>🔒<br>Shooting</div>
  <div id='sq-4' style='width:105px; height:50px; border:2px solid #4400aa; border-radius:5px; display:flex; align-items:center; justify-content:center; font-size:11px; color:#553388; background:#0d0020; text-align:center; padding:4px;'>🔒<br>Boss & Gravity</div>
  <span id='kill-counter' style='margin-left:auto; color:#aaaaff; font-size:12px; opacity:0.7;'>0 kills</span>
</div>
<div id='lesson-progress' style='font-family: monospace; background: #0a001a; border: 1px solid #4400aa; padding: 10px 18px; border-radius: 6px; color: #aaaaff; font-size: 13px; margin-bottom: 8px;'>
  🔒 <strong>Lesson locked.</strong> Destroy 5 enemies in-game to reveal the first concept.
  <br><span style='opacity:0.6; font-size:12px;'>Keep shooting — each 5 kills unlocks the next section.</span>
</div>


<div class='lesson-block' id='lesson-4' style='display:none'>
<hr />
<h2>🔒 Enemy Boss, Chasing System &amp; Gravity Asteroids <em>(unlock: 20 kills)</em></h2>

<!-- Box 1: Boss Enemy -->
<div style='background:#0d0022; border:1px solid #4400aa; border-radius:8px; padding:16px 20px; margin-bottom:16px;'>
<h3 style='margin-top:0; color:#cc88ff;'>Boss Alien</h3>
<p>On wave 3 a tanky <strong>Boss Alien</strong> spawns at the top of the screen. Unlike regular enemies, it actively homes in on the player's ship each frame. The trick is <code>Math.atan2</code>, which converts an (x, y) displacement into an angle — then <code>cos</code>/<code>sin</code> turn that angle back into a unit direction vector:</p>
<pre><code class="language-javascript">function updateBoss(boss, ship) {
  const dx = ship.x - boss.x;
  const dy = ship.y - boss.y;

  // atan2 gives the angle from boss to ship — dy goes first, easy to mix up!
  const angle = Math.atan2(dy, dx);

  // Step toward the ship along that angle
  boss.x += Math.cos(angle) * boss.speed;
  boss.y += Math.sin(angle) * boss.speed;
}
</code></pre>
<p>While the boss is alive, <code>worldSpeed</code> drops to <code>0.4</code>, slowing every asteroid and regular enemy automatically — creating a dramatic slowdown effect without any extra conditional logic. Killing the boss restores full speed and counts as 5 kills. Each successive boss generation is faster and has a new color palette.</p>
</div>

<!-- Box 2: Chase Logic on Regular Enemies -->
<div style='background:#0d0022; border:1px solid #4400aa; border-radius:8px; padding:16px 20px; margin-bottom:16px;'>
<h3 style='margin-top:0; color:#cc88ff;'>Chasing Enemies</h3>
<p>Starting on wave 2, some regular enemies become <strong>chasers</strong> that lock onto the ship instead of drifting straight down. The core formula: divide the displacement vector by its length to isolate direction (a unit vector), then scale by speed:</p>
<pre><code class="language-javascript">function updateChaser(e, ship) {
  const dx   = ship.x - e.x;
  const dy   = ship.y - e.y;
  const dist = Math.hypot(dx, dy) || 1;  // avoid divide-by-zero

  // Ramp up speed gradually — chasers accelerate as they close in
  e.chaseAcc = Math.min(e.speed, e.chaseAcc + 0.0006);
  const spd  = (e.speed + e.chaseAcc) * worldSpeed;

  e.x += (dx / dist) * spd;  // move along the unit vector
  e.y += (dy / dist) * spd;

  // Rotate the enemy sprite to face the ship
  e.angle = Math.atan2(dy, dx);
}
</code></pre>
<p>Chasers are drawn as arrow shapes and rotated via <code>ctx.rotate(e.angle + Math.PI / 2)</code> so they visually point at the ship. The proportion of chasers scales with wave number — later waves are far more aggressive.</p>
</div>

<!-- Box 3: Gravity Asteroids -->
<div style='background:#0d0022; border:1px solid #4400aa; border-radius:8px; padding:16px 20px; margin-bottom:16px;'>
<h3 style='margin-top:0; color:#cc88ff;'>Gravity-Driven Asteroids</h3>
<p>Asteroids used to fall at a constant speed. A per-frame gravity loop makes them <em>accelerate</em> as they fall, exactly like real objects under gravity. Each asteroid tracks a <code>verticalVelocity</code> and a <code>gravityAcceleration</code>:</p>
<pre><code class="language-javascript">function updateAsteroid(a) {
  // Subtract gravity each frame — velocity grows in magnitude (downward)
  a.verticalVelocity -= a.gravityAcceleration;

  // Cap at terminal velocity so asteroids stay readable at any wave count
  if (-a.verticalVelocity > a.terminalVelocity) {
    a.verticalVelocity = -a.terminalVelocity;
  }

  // Flip sign: canvas y increases downward, but velocity is stored as negative
  a.y += -a.verticalVelocity * worldSpeed;

  // Spin the asteroid as it falls
  a.rotation += a.spinSpeed;
}
</code></pre>
<p>Why store velocity as negative? Canvas y grows <em>downward</em>, so a positive "fall" direction is negative in the velocity math — flipping the sign on the final move keeps the gravity formula intuitive (<code>v -= g</code> means pulling down).</p>
<p>The <code>worldSpeed</code> multiplier automatically slows asteroids during the boss fight without any extra code — the same variable that slows regular enemies slows gravity too.</p>
</div>

<p style='margin-top:24px; padding:12px 16px; background:#0d2b00; border:1px solid #00cc44; border-radius:6px; color:#44ff88; font-family:monospace; font-size:13px;'>
🎉 <strong>Lesson complete!</strong> You've seen every core system in Void Striker — parallax layers, scene switching, triple-shot bullets, character config arrays, boss pursuit with atan2, normalized chase vectors, and gravity physics.</p>

<p style='margin-top:14px; padding:14px 18px; background:#1a0d2b; border:1px solid #aa44ff; border-radius:6px; color:#dda0ff; font-family:monospace; font-size:13px; text-align:center;'>
✨ <strong>Credits</strong> — <em>Team Sorcerers</em> for the <strong>Gravity System</strong> &nbsp;•&nbsp; the <em>Ocean Team</em> for the <strong>Boss Chasing Enemy</strong> &nbsp;•&nbsp; <em>Lance Oberiano, Yiming Yin &amp; Arjun Ganesh</em> for the <strong>Chase Logic &amp; Character Swap</strong> ✨
</p>
</div>


<script>
(function() {
  const milestones = [5, 10, 15, 20];
  let unlocked = 0;

  function updateSquare(n, label) {
    var sq = document.getElementById('sq-' + n);
    if (sq) {
      sq.style.background = '#0d2b00';
      sq.style.borderColor = '#00cc44';
      sq.style.color = '#44ff88';
      sq.innerHTML = '&#x2705;<br>' + label;
    }
  }

  const labels = ['Parallax BG', 'BG Change', 'Shooting', 'Boss & Gravity'];

  function unlock(n) {
    var block = document.getElementById('lesson-' + n);
    if (block) {
      block.style.display = 'block';
      block.style.animation = 'fadeIn 0.6s ease';
      if (typeof hljs !== 'undefined') {
        block.querySelectorAll('pre code').forEach(function(el) {
          hljs.highlightElement(el);
        });
      }
    }
    var h2s = document.querySelectorAll('h2');
    h2s.forEach(function(h) {
      const label = labels[n-1];
      if (h.textContent.includes(label)) {
        h.textContent = h.textContent.replace('🔒', '✅');
      }
    });
    updateSquare(n, labels[n-1]);
    var progress = document.getElementById('lesson-progress');
    if (progress) {
      if (n < 4) {
        progress.innerHTML = '✅ ' + labels[n-1] + ' unlocked! Destroy ' + milestones[n] + ' total enemies for the next one.';
      } else {
        progress.innerHTML = '🎉 <strong>All sections unlocked!</strong> You finished the Void Striker lesson.';
      }
    }
  }

  window.addEventListener('vs-kills', function(e) {
    var total = e.detail && e.detail.total;
    if (!total) return;
    var counter = document.getElementById('kill-counter');
    if (counter) counter.textContent = total + ' kill' + (total === 1 ? '' : 's');
    milestones.forEach(function(m, idx) {
      if (total >= m && unlocked <= idx) {
        unlocked = idx + 1;
        unlock(idx + 1);
      }
    });
  });

  var style = document.createElement('style');
  style.textContent = '@keyframes fadeIn { from { opacity: 0; transform: translateY(10px); } to { opacity: 1; transform: translateY(0); } }';
  document.head.appendChild(style);
})();
</script>
